<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ChemNLP_TitleToAbstract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[ChemNLP](https://github.com/usnistgov/chemnlp) is a Natural Language Processing (NLP) based library for materials chemistry text data. Reference [here](https://doi.org/10.1021/acs.jpcc.3c03106). ChemNLP can be used for (1) curating open access datasets for materials and chemistry literature, developing and comparing traditional machine learning, transformers and graph neural network models for (2) classifying and clustering texts, (3) named entity recognition for large-scale text-mining, (4) abstractive summarization for generating titles of articles from abstracts, (5) text generation for suggesting abstracts from titles, (6) integration with density functional theory dataset for identifying potential candidate materials such as superconductors, and (7) web-interface development for text and reference query.


Lets install a few packages first

In [ ]:
%%time
!pip install -q transformers datasets evaluate rouge_score jarvis-tools

In [ ]:
%%time
import os
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -q -e .

In [ ]:
%%time
from transformers import Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
import torch
import math
from tqdm import tqdm
import time
import evaluate
from collections import defaultdict
from jarvis.db.jsonutils import dumpjson
import random
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

random_seed = 123
torch.manual_seed(random_seed)
random.seed(0)
np.random.seed(random_seed)
torch.backends.cudnn.deterministic = True

rouge_score = evaluate.load("rouge")

# import torch
# torch.cuda.is_available = lambda : False

tqdm.pandas()
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")


Load openai-gpt2 model from [huggingface](https://huggingface.co/openai-community/gpt2-medium)

In [ ]:
%%time
# prompt="Nonuniform superconductivity and Josephson effect in conical ferromagnet can be described as"
model_checkpoint = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_checkpoint).to(device)
def generate_text(prompt="What is a superconductor?",max_new_tokens=250,model_checkpoint = "gpt2-medium"):
    # tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
    # model = AutoModelForCausalLM.from_pretrained(model_checkpoint).to(device)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    # model = AutoModelForCausalLM.from_pretrained(checkpoint)
    # outputs = model.generate(**inputs, do_sample=True)
    outputs = model.generate(
        **inputs, do_sample=True, max_new_tokens=max_new_tokens
    )
    out = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return (out[0].replace('\n',' '))

In [ ]:
%%time
p = generate_text(prompt="What is a superconductor?")
#print(p)

In [ ]:
p

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
!ls jarvis_leaderboard/benchmarks/AI/TextGen/arxiv_gen_text.json.zip

Let's generate article title to abstract for a few condensed matter physics, superconductor articles

In [ ]:
import json,zipfile
fname = 'jarvis_leaderboard/benchmarks/AI/TextGen/arxiv_gen_text.json.zip'
temp = 'arxiv_gen_text.json'
zp = zipfile.ZipFile(fname)
train_test = json.loads(zp.read(temp))

In [ ]:
from jarvis.db.figshare import data
import pandas as pd
arxiv_summary = data('arxiv_summary')
df = pd.DataFrame(arxiv_summary)
df = df.drop_duplicates()

We will use the prompt `$title` can be described as `$abstract`

In [ ]:
%%time

from tqdm import tqdm
info = {}
for i,j in tqdm(train_test['test'].items()):
  title=df[df['id']==i]['text'].values[0]
  actual_abstract =df[df['id']==i]['ctext'].values[0]
  prompt = title+' can be described as '
  #prompt = 'Write an abstract on the title '+ prompt
  pred = generate_text(prompt)
  info[i] = pred
  #break

In [ ]:
from jarvis.db.jsonutils import dumpjson
dumpjson(data=info,filename='textgen.json')

In [ ]:
mem=[]
for i,j in info.items():
  info1={}
  info1['id']=str(i)
  info1['prediction']=j
  info1['target']=df[df['id']==i]['ctext'].values[0].split('summarize:')[1]
  mem.append(info1)


In [ ]:
import pandas as pd
dff=pd.DataFrame(mem)
dff

In [ ]:
dff.to_csv('AI-TextGen-text-arxiv_gen-test-rouge.csv',index=False)



In [ ]:
!zip AI-TextGen-text-arxiv_gen-test-rouge.csv.zip AI-TextGen-text-arxiv_gen-test-rouge.csv

In [ ]:
from rouge import Rouge
rouge = Rouge()
hypothesis = dff["prediction"]
reference = dff["target"]
scores = rouge.get_scores(hypothesis, reference, avg=True)["rouge-1"]["r"]
print("rouge scores", scores)

See benchmark here: https://pages.nist.gov/jarvis_leaderboard/AI/TextGen/